# Anime Face GAN — Colab Training
Run all cells in order. Checkpoints save to Google Drive every 10 epochs.
For the full architecture walkthrough see `notebook.ipynb`.

In [ ]:
import sys, os, subprocess

if not os.path.exists('/content/anime-face-gan'):
    subprocess.run(['git', 'clone', 'https://github.com/xavier-oc-programming/anime-face-gan',
                    '/content/anime-face-gan'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='/content/anime-face-gan', check=True)

os.chdir('/content/anime-face-gan')

# Only install kagglehub — TensorFlow, numpy, PIL etc are pre-installed on Colab.
# Installing the full requirements.txt downgrades numpy to <2.0 which conflicts
# with Colab's TensorFlow binaries and causes a binary incompatibility crash.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub>=0.3'], check=True)
print('Ready.')

In [ ]:
# Download dataset
import shutil, kagglehub
from pathlib import Path
from config import DATA_DIR

existing = list(DATA_DIR.glob('*.jpg')) + list(DATA_DIR.glob('*.png'))
if existing:
    print(f'Dataset already present — {len(existing):,} images')
else:
    src = Path(kagglehub.dataset_download('splcher/animefacedataset'))
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for p in src.rglob('*.jpg'):
        shutil.copy(p, DATA_DIR / p.name)
    print(f'Copied {len(list(DATA_DIR.glob("*.jpg"))):,} images → {DATA_DIR}')

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
from pathlib import Path
from config import SAVE_INTERVAL

drive.mount('/content/drive')
model_dir   = Path('/content/drive/MyDrive/anime-face-gan/models')
samples_dir = Path('/content/drive/MyDrive/anime-face-gan/samples')
print(f'Checkpoints → {model_dir}')
print(f'Saved every {SAVE_INTERVAL} epochs — a crash loses at most one interval.')

In [ ]:
# Train
from train import train
train(model_dir=model_dir, samples_dir=samples_dir)

In [ ]:
# Download generator.keras and training_log.json from Drive after training
from google.colab import files
files.download(str(model_dir / 'generator.keras'))
files.download(str(model_dir / 'training_log.json'))